In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "ministral-3:14b-instruct-2512-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

# Verificar integridad
assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conexión local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Consultar características reales del modelo
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: ministral-3:14b-instruct-2512-q4_K_M
Identificador: 4760c35aeb9d
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'vision', 'tools']
Thinking: None
Parámetros: temperature                    0.15

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-MIN3-14B-01"

# 1. Mismo diagnóstico de los modelos anteriores
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Recuperar configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Variante Instruct sin razonamiento configurable
solicitud.pop("think", None)

# Configuración comparable con Ministral 3 8B
solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 1200,
    "temperature": 0
}

# Mantener los prompts V3 originales
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

assert configuracion["prompt_version"] == "V3"
assert "completion" in detalles.get("capabilities", [])
assert "thinking" not in detalles.get("capabilities", [])

# 3. Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "ministral3_14b_directa_v1",
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico en Ollama local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar sin sobrescribir resultados
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar resultados
print("Caso:", CASO_ID)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-MIN3-14B-01
Estado: ok
Tiempo cliente: 29.831 s
Tokens entrada: 364
Tokens generados: 1084
Terminación: stop
Razonamiento separado: False
total_duration 29.829 s
load_duration 4.157 s
prompt_eval_duration 0.193 s
eval_duration 25.472 s

RESPUESTA FINAL:

### **Evidencia**
La única evidencia disponible en este caso sintético es la siguiente:
- **[E1]**: El **Autor X** ha publicado un artículo sobre **bases de datos académicas**, un tema vinculado a la gestión, organización o acceso a información estructurada en contextos científicos.
- **[E2]**: El **Autor Y** ha publicado un artículo sobre **recuperación de información**, un campo que aborda técnicas para extraer, indexar o recuperar datos relevantes de fuentes (incluyendo bases de datos).

No existen registros de colaboración previa entre ambos autores, ni se documentan otros factores (ej.: proyectos compartidos, afiliaciones institucionales comunes, citas cruzadas en sus trabajos o métricas de coautoría histórica).

---

#

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "ministral-3:14b-instruct-2512-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line) for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-MIN3-14B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert "think" not in diagnostico["solicitud"]

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 1200,
    "temperature": 0
}

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo instalado
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar ejecuciones
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "ministral3_14b_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar la solicitud del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir únicamente el caso y su evidencia
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "ministral3_14b_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada caso inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 1180 tokens | 29.187 s
S-006 | ok | 849 tokens | 19.999 s
S-007 | ok | 998 tokens | 23.592 s
S-008 | truncada | 1200 tokens | 28.293 s
S-009 | ok | 1015 tokens | 23.995 s
S-010 | ok | 1043 tokens | 24.786 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "ministral-3:14b-instruct-2512-q4_K_M"
API = "http://127.0.0.1:11434"
CONFIG_RESCATE = "ministral3_14b_rescate_2048"

# 1. Recuperar el caso truncado
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line)
        for line in f if line.strip()
    ]

assert not any(
    r.get("case_id") == "S-008"
    and r.get("configuracion_modelo") == CONFIG_RESCATE
    for r in registros
), "El rescate ya está registrado."

anterior = next(
    r for r in registros
    if r["case_id"] == "S-008"
    and r.get("configuracion_modelo") == "ministral3_14b_directa_v1"
)

assert anterior["estado"] == "truncada"
assert anterior["respuesta_original"]["done_reason"] == "length"

# 2. Mantener solicitud original
solicitud = copy.deepcopy(anterior["solicitud"])

assert solicitud["model"] == MODELO
assert solicitud["options"]["num_ctx"] == 4096
assert solicitud["options"]["temperature"] == 0
assert solicitud["options"]["num_predict"] == 1200

# Cambiar solamente el límite de generación
solicitud["options"]["num_predict"] = 2048

# 3. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_actual = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_actual["digest"] == anterior["modelo_digest"]

# 4. Preparar registro adicional
registro = {
    "execution_id": str(uuid4()),
    "case_id": "S-008",
    "tipo": anterior["tipo"],
    "grupo": "comparacion",
    "configuracion_modelo": CONFIG_RESCATE,
    "referencia_execution_id": anterior["execution_id"],
    "motivo": "Truncamiento con 1200 tokens",
    "prompt_version": "V3",
    "modelo_digest": anterior["modelo_digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 1
}

inicio = time.perf_counter()

# 5. Ejecutar nuevamente S-008
try:
    peticion = Request(
        API + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar sin eliminar el truncamiento original
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar resultados
print("Caso:", registro["case_id"])
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Caso: S-008
Estado: ok
Tiempo cliente: 23.306 s
Tokens entrada: 388
Tokens generados: 924
Terminación: stop

RESPUESTA FINAL:

### **Evidencia**
La evidencia disponible consiste en dos artículos independientes:
- **[E1]**: *Artículo P-O01* (Autor O, 2024), centrado en **optimización de programación energéticamente eficiente en centros de datos con procesadores heterogéneos**. Pertenece al dominio de la **informática de alto rendimiento, gestión de recursos y eficiencia energética**.
- **[E2]**: *Artículo P-P01* (Autor P, 2025), enfocado en **segmentación de imágenes retinales mediante redes neuronales convolucionales**. Se ubica en el campo de la **visión por computadora y procesamiento de imágenes médicas**.

No existe **evidencia directa** que vincule a los autores en un proyecto conjunto, colaboración previa o similitud temática entre sus publicaciones. Los abstracts no comparten metodologías, objetivos o áreas de investigación comunes, y las fechas de publicación (2024 vs. 2025) re